## Devoir Maison “Econometric Evaluation of Public Policies”

Groupe 4 - François Bradesi, Fanny Daubet, Titouan Morvan, Raphaël Thireau

In [ ]:
# Imports et lecture du jeu de données
import pandas as pd
import numpy as np
import pyreadr
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import norm

result = pyreadr.read_r('df_group4.Rda')
df = result['df_group4']

### 1. Early stage analysis

Question 1: Given the description of the policy, build a variable Gi equal to 1 if municipality i belongs to a treated province and 0 otherwise. Run a descriptive statistic analysis to compare the groups {G = 0} and {G = 1}. Are these groups very different in terms of outcomes? If so, is it a problem? Are these groups very different in terms of the other variables? If so, is it a problem?

In [11]:
## Question 1

# Variable Gi valant 1 si la commune i appartient à une province traitée et 0 sinon
treated_provinces = [2, 3, 6]
df['G'] = df['province_index'].isin(treated_provinces).astype(int)

# Satistiques descriptives pour comparer les groupes {G = 0} et {G = 1}

### 2. Analysis without covariates

#### 2.1 Pseudo-parallel trend tests

#### 2.2 Average treatment effect estimation

### 3. Analysis with covariates
The two variables urban and organic_school_canteen can serve as covariates here.

Question 7: Write down the parallel trend assumptions required to identify and estimate the ATTs when one uses a DID approach with covariates. What is the other assumption needed to run a DID analysis with covariates? Verify this second assumption in the data.

In [ ]:
# Lorsqu'on utilise une approche DID avec covariables, l'hypothèse requise est celle des tendances parallèles conditionnelles.

# Formellement, pour chaque groupe de traitement 𝑔 et chaque période 𝑡 :
# E[𝑌𝑡(0)−𝑌𝑡-1(0) ∣ 𝐺=1, 𝑋] = E[𝑌𝑡(0)−𝑌𝑡-1(0) ∣ 𝐺=0, 𝑋]

# Où :
# 𝑌𝑡(0) représente le résultat potentiel sans traitement,
# 𝐺 indique le statut de traitement,
# 𝑋 représente les covariables (ici : urban_canteen et organic_school_canteen).

# L'autre hypothèse clé est :
# S(P_{X|G=1}) ⊂ S(P_{X|G=0})

# où S(P) représente le support de la loi P.

# Vérification de l'hypothèse de support :
urban_tab = pd.crosstab(df['G'], df['urban'])
organic_tab = pd.crosstab(df['G'], df['organic_school_canteen'])

print(urban_tab)
print(organic_tab)

# Les valeurs des variables urban et organic_school_canteen prises par les communes dans le groupe de contrôle sont les mêmes que celles prises par celles du groupe traité.
# Les supports des lois conditionnelles sont donc identiques, ce qui valide l'hypothèse.

urban     0     1
G                
0      2058   681
1       940  1321
organic_school_canteen   0.0  1.0
G                                
0                       1907  832
1                       1453  808


Question 8: Let Xi stand for a vector stacking the variables urbani and organic_school_canteeni for
municipality i. A classmate of yours suggests to run the following regression to recover the ATTs
(using years 2017, 2018 and 2019):

ln_avg_household_organicit = α + βGi + γ1{t ≥ 2018} + δGi × 1{t ≥ 2018} + Xi'μ + εit. (1)

Do you think this regression is well-suited to identify both the ATTs in 2018 and 2019? (hint:
what may be the limitation of this regression if the ATTs at the two dates are not equal?) Which
coefficient of this regression is supposed to capture treatment effects? Estimate the ATTs with
this regression and build a 95% confidence interval (do not forget to cluster standard errors at
the municipality level).

In [75]:
# Cette spécification identifie uniquement un effet moyen post-traitement et suppose que :
# ATT2018​ = ATT2019
# La modèle proposé est donc mal spécifié si les effets diffèrent entre 2018 et 2019.
# Le coefficient d’intérêt est δ (interaction G × post).

In [ ]:
# Création d'un identifiant de municipalité
df['municipality_id'] = df.index

# Transformation des données en panel
df_long = df.melt(
    id_vars=['municipality_id', 'province_index', 'urban', 'organic_school_canteen', 'G'],
    value_vars=[
        'ln_avg_household_organic_2016',
        'ln_avg_household_organic_2017',
        'ln_avg_household_organic_2018',
        'ln_avg_household_organic_2019'
    ],
    var_name='year',
    value_name='ln_avg_household_organic'
)

# Extraction de l'année
df_long['year'] = df_long['year'].str[-4:].astype(int)

# Variable post
df_long['post'] = (df_long['year'] >= 2018).astype(int)

# Modèle
model = smf.ols(
    'ln_avg_household_organic ~ G + post + G:post + urban + organic_school_canteen',
    data=df_long
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_long['municipality_id']}
)

print(model.summary())

# Dans notre régression, l'ATT estimé est égal à 0,8812 (erreur standard 0,016),
# ce qui indique une augmentation statistiquement significative de la consommation de produits biologiques
# par les ménages dans les municipalités traitées après la mise en œuvre de la politique.

                               OLS Regression Results                               
Dep. Variable:     ln_avg_household_organic   R-squared:                       0.616
Model:                                  OLS   Adj. R-squared:                  0.616
Method:                       Least Squares   F-statistic:                 1.597e+04
Date:                      Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                              09:34:45   Log-Likelihood:                -10207.
No. Observations:                     20000   AIC:                         2.043e+04
Df Residuals:                         19994   BIC:                         2.047e+04
Df Model:                                 5                                         
Covariance Type:                    cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------

In the next two questions, you are expected to estimate separately the two ATTs using the OR approach introduced in class in order to obtain alternative treatment effect estimates to what was obtained with Model (1). As a reminder, the OR approach requires estimating a mt(·) function (see the slides for definitions).

Question 9: Estimate the mt(·) function using a linear regression approach. Build a 95% confidence interval for each ATT (you may have to resort to the bootstrap). Comment on the results and compare with what you obtained without covariates.

In [ ]:
# ***** Approche OR (Outcome Regression) pour estimer l'ATT *****

# On suppose que m(x) = x'θ (on note ' la transposée d'une matrice quelconque).
# On peut alors écrire dans le groupe de contrôle que :
# Y1 − Y0 = X'θ + U, avec E[U | X, D1 = 0] = 0

# On peut estimer θ en effectuant une régression linéaire de Y1 − Y0 sur X dans le groupe témoin.
# ˆθ est ainsi l'estimateur des moindres carrés ordinaires (MCO) et on estime m(x) par ^m(x) = x'ˆθ.

# L'identification OR de l'ATT est : ATT = E[Y1 − Y0 | D1 = 1] − E[m(X) | D1 = 1].
# Cette expression est ensuite estimée par la moyenne des différences entre les résultats observés dans le groupe traité
# et les résultats prédits à partir du modèle de régression dans le groupe de contrôle
# (cet estimateur est consistant par la loi forte des grands nombres) :
# ^ATT = ^E[Y1 − Y0 | D1 = 1] − ^E[^m(X) | D1 = 1] (contrepartie empirique de l'ATT).

# Calcul de la différence de résultats
df['dY_2018'] = df['ln_avg_household_organic_2018'] - df['ln_avg_household_organic_2017']
df['dY_2019'] = df['ln_avg_household_organic_2019'] - df['ln_avg_household_organic_2017']

Xvars = ['urban','organic_school_canteen']

def OR_ATT(data, dY):
    
    controls = data[data['G']==0]
    treated = data[data['G']==1]
    
    Xc = sm.add_constant(controls[Xvars])
    Xt = sm.add_constant(treated[Xvars])
    
    model = sm.OLS(controls[dY], Xc).fit()
    
    mhat = model.predict(Xt)
    
    ATT = treated[dY].mean() - mhat.mean()
    
    return ATT

# ATT estimés
ATT_2018 = OR_ATT(df,'dY_2018')
ATT_2019 = OR_ATT(df,'dY_2019')

print("ATT 2018:", ATT_2018)
print("ATT 2019:", ATT_2019)


# ****** Bootstrap pour l'intervalle de confiance à 95% de l'ATT ******

# L'^ATT est asymptotiquement normal (sans biais + théorème central limite).
# Il converge donc vers une loi normale N(0,σ²) avec σ² > 0.

# En considérant les quantiles de la loi normale centrée réduite Q(1-α/2) et Q(α/2) (par symétrie Q(α/2) = -Q(1-α/2)),
# on peut construire un intervalle de confiance à 95% pour l'ATT en utilisant sur un estimateur
# ^σ de σ :
# [^ATT - (Q(1-α/2) * ^σ)/ √n, ^ATT + (Q(1-α/2) * ^σ)/ √n]

# Le bootstrap permet de remplacer ^σ/√n par l'écart-type de la distribution empirique des estimateurs obtenus à partir d'échantillons bootstrapés.

B = 1000
boot_2018 = []
boot_2019 = []

for b in range(B):
    
    sample = df.sample(len(df), replace=True)
    
    boot_2018.append(OR_ATT(sample,'dY_2018'))
    boot_2019.append(OR_ATT(sample,'dY_2019'))

ci_2018 = np.percentile(boot_2018,[2.5,97.5])
ci_2019 = np.percentile(boot_2019,[2.5,97.5])

print("95% CI ATT 2018:", ci_2018)
print("95% CI ATT 2019:", ci_2019)

# Les deux effets sont positifs et statistiquement significatifs.
# Cela suggère que la politique a augmenté le niveau moyen de consommation de produits biologiques des ménages dans les municipalités concernées par rapport à l'évolution contrefactuelle prévue à partir du groupe de contrôle.
# L'ampleur de l'effet est plus importante en 2018 qu'en 2019, ce qui indique que l'effet de la mesure diminue avec le temps.

ATT 2018: 1.1690198009618844
ATT 2019: 0.5918787312661007
95% CI ATT 2018: [1.12660219 1.21260274]
95% CI ATT 2019: [0.56824353 0.61618516]


Question 10: Estimate the mt(·) function using a fully nonparametric approach (hint: you only have two binary covariates so this is very straightforward if you look carefully at the slides). Build a 95% confidence interval for each ATT (you may have to resort to the bootstrap). Comment on the results and compare with what you obtained without covariates.

In [ ]:
# ***** Estimation OR non paramétrique de l'ATT *****

# Ici, X est un vecteur qui ne contient que des variables catégorielles (urban et organic_school_canteen).
# On peut donc estimer m(x) de manière non paramétrique par la contrepartie empirique de :
# E[ Y1 - Y0 | X = x, D1 = 0].
# Donc ^ATT = ^E[ Y1 - Y0 | X = x, D1 = 0]
# Cet estimateur est consistant par la loi forte des grands nombres (valable par l'hypothèse d'indépendance des muncipalités
# et car les variables Yi1, Yi0 et Di1 sont intégrables car bornées).

# Estimation non paramétrique de m(x) :
def OR_nonparam_ATT(data, dY):

    controls = data[data['G']==0]
    treated = data[data['G']==1]

    # Moyenne par cellule X dans le groupe contrôle
    mhat = controls.groupby(['urban','organic_school_canteen'])[dY].mean()

    # Prédiction pour les traités
    treated_cells = treated[['urban','organic_school_canteen']]
    mhat_treated = treated_cells.merge(
        mhat.rename('mhat'),
        left_on=['urban','organic_school_canteen'],
        right_index=True
    )['mhat']

    ATT = treated[dY].mean() - mhat_treated.mean()

    return ATT

# Estimation des ATT non paramétriques
ATT_2018_np = OR_nonparam_ATT(df,'dY_2018')
ATT_2019_np = OR_nonparam_ATT(df,'dY_2019')

print("ATT 2018:", ATT_2018_np)
print("ATT 2019:", ATT_2019_np)

# ***** Bootstrap pour IC à 95% *****
# On applique un raisonnement analogue à celui de la question précédente :

B = 1000
boot_2018 = []
boot_2019 = []

for b in range(B):

    sample = df.sample(len(df), replace=True)

    boot_2018.append(OR_nonparam_ATT(sample,'dY_2018'))
    boot_2019.append(OR_nonparam_ATT(sample,'dY_2019'))

ci_2018 = np.percentile(boot_2018,[2.5,97.5])
ci_2019 = np.percentile(boot_2019,[2.5,97.5])

print("95% CI 2018:", ci_2018)
print("95% CI 2019:", ci_2019)

# Les résultats étant similaires aux estimations OR linéaires, cela suggère que la spécification linéaire pour m(x) est adéquate.

ATT 2018: 1.1689271308473068
ATT 2019: 0.5918161145830401
95% CI 2018: [1.12713361 1.21366233]
95% CI 2019: [0.56814396 0.61556957]


Question 11: For this last question, focus on years 2017 and 2018. The goal is to detect potential heterogeneity of treatment effects in year 2018 (still using a DID approach with years 2017 and 2018). Give the definition of the CATT function in the present context. Propose a treatment effect heterogeneity test based on the CATT function (several possibilities are proposed in the slides, choose the one you prefer). Implement this test and comment on the result.

In [ ]:
# ***** Définition de la fonction CATT (Conditional Average Treatment Effect on the Treated) *****

# La Conditional Average Treatment Effect on the Treated (CATT) mesure l’effet du traitement conditionnellement aux covariables .
# Dans notre contexte (municipalités, deux périodes : 2017 = pré, 2018 = post) :
# CATT(x) = E[Y2018(1)−Y2018(0) ∣ D1=1, X=x].
# Donc le CATT(x) donne l’effet attendu du traitement pour les municipalités traitées ayant les caractéristiques  X=x.


# ***** Tests d'hétérogénéité de l'effet du traitement *****

# On souhaite tester l'hypothèse d'égalité des CATT pour tout couple x,y appartenant au support de la loi conditionnelle P_{X|D1=1} dans le groupe traité :
# Formellement : soit x,y ∈ S(P_{X|D1=1})
# on propose le test H0 : CATT(x) = CATT(y) vs H1 : tels que CATT(x) ≠ CATT(y)
# Intuitivement, le test va consister à comparer les contreparties empiriques de CATT(x) et CATT(y)
# Le test sera rejeté si la différence en valeur absolue est plus grande qu'un certain seuil
# (valeur du seuil : QN(0,1)(1 − α/2) * √(w'^Σw) / √n)

# D'après le théorème central limite multivarié, le vecteur de la distribution de la différence entre les estimateurs de CATT(x) et CATT(y) converge vers une loi normale multivariée N(0, Σ)
# où Σ est la matrice de covariance asymptotique.

# Configuration
cells = [(0,0), (0,1), (1,0), (1,1)]
B = 1000
n = len(df)
CATT_boot_list = []

# Bootstrap pour obtenir la distribution des CATT
for b in range(B):
    sample = df.sample(n, replace=True)
    
    # Moyennes par strate
    m0 = sample[sample['G']==0].groupby(['urban','organic_school_canteen'])['dY_2018'].mean()
    m1 = sample[sample['G']==1].groupby(['urban','organic_school_canteen'])['dY_2018'].mean()
    
    # reindex est crucial pour éviter les KeyError et NaNs lors du calcul
    catt_sample = (m1 - m0).reindex(cells)
    CATT_boot_list.append(catt_sample.values)

# Matrice de covariance des B estimateurs de CATT pour les 4 cellules
boot_matrix = np.array(CATT_boot_list)
boot_matrix = pd.DataFrame(boot_matrix).bfill().ffill().values
cov_matrix = np.cov(boot_matrix, rowvar=False)
mean_catt = np.nanmean(boot_matrix, axis=0)

# Affichage des résultats
print("Valeurs estimées des CATT par profil de covariables :")
print("-" * 55)
for i, cell in enumerate(cells):
    status_urban = "Urbain" if cell[0] == 1 else "Rural"
    status_canteen = "Cantine Bio" if cell[1] == 1 else "Pas de Cantine Bio"
    
    print(f"Profil {cell} ({status_urban:<6} | {status_canteen:<18}) : {mean_catt[i]:>8.4f}")
print("-" * 55)


# Tests de Student (paires)
print(f"{'Comparaison':<25} | {'Différence':<12} | {'Z-stat':<8} | {'p-value':<8}")
print("-" * 65)

n_cells = len(cells)
for i in range(n_cells):
    for j in range(i + 1, n_cells):
        diff = mean_catt[i] - mean_catt[j]
        var_diff = cov_matrix[i, i] + cov_matrix[j, j] - 2 * cov_matrix[i, j]
        z_stat = diff / np.sqrt(var_diff)
        p_val = 2 * (1 - norm.cdf(np.abs(z_stat)))
        
        label = f"{cells[i]} vs {cells[j]}"
        print(f"{label:<25} | {diff:>12.4f} | {z_stat:>8.2f} | {p_val:>8.4f}")

# Les fonctions CATT estimées révèlent une forte hétérogénéité de l'effet du traitement :
# les communes urbaines traitées connaissent une augmentation importante de la consommation de produits biologiques des ménages,
# tandis que les communes rurales ne semblent présenter aucun effet.
# La présence d'une cantine scolaire proposant des produits biologiques ne semble pas avoir d'incidence notable sur cet effet.

# Les tests F confirment que cette hétérogénéité est statistiquement significative :
# toutes les p-values comparant un profil rural à un profil urbain sont à 0,0000 avec des $Z$-stats très élevées ($> 80$).
# L'hétérogénéité géographique est donc indiscutable.
# A l'inverse, la présence d'une cantine scolaire n'a aucun effet significatif :
# la p-value pour la comparaison entre les profils ruraux et urbains concernant la Cantine Bio est de 0,3173 (non significatif).
# et la p-value pour la comparaison entre les profils urbains est de 0,6874 (non significatif).

Valeurs estimées des CATT par profil de covariables :
-------------------------------------------------------
Profil (0, 0) (Rural  | Pas de Cantine Bio) :  -0.0020
Profil (0, 1) (Rural  | Cantine Bio       ) :   0.0131
Profil (1, 0) (Urbain | Pas de Cantine Bio) :   2.0031
Profil (1, 1) (Urbain | Cantine Bio       ) :   1.9938
-------------------------------------------------------
Comparaison               | Différence   | Z-stat   | p-value 
-----------------------------------------------------------------
(0, 0) vs (0, 1)          |      -0.0151 |    -1.04 |   0.2968
(0, 0) vs (1, 0)          |      -2.0052 |  -166.57 |   0.0000
(0, 0) vs (1, 1)          |      -1.9958 |   -92.63 |   0.0000
(0, 1) vs (1, 0)          |      -1.9900 |  -125.78 |   0.0000
(0, 1) vs (1, 1)          |      -1.9807 |   -83.42 |   0.0000
(1, 0) vs (1, 1)          |       0.0093 |     0.42 |   0.6768
